In [3]:
import glob, json, math, os, time
import torch
import torch.nn.functional as F
from einops import rearrange, einsum
from safetensors.torch import load_file
from tokenizers import Tokenizer
from tqdm import tqdm, trange
from torch.distributions import Categorical

# Loading Model + Setup

In [4]:
max_tot_tokens = 1024 # small, actually sliding window and regular don't differ for this size
checkpoint = "checkpoints/diffusiongemma-26B-A4B-it"
canvas_len = 256

In [5]:
torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1)))
torch.set_default_dtype(torch.bfloat16)

In [ ]:
model_config = json.load(open(f"{checkpoint}/config.json"))['text_config']
gen_config = json.load(open(f"{checkpoint}/generation_config.json"))

model_config # describes details of the model

{'confidence_threshold': 0.005,
 'eos_token_id': [1, 106, 50],
 'max_denoising_steps': 48,
 'max_new_tokens': 256,
 'pad_token_id': 0,
 'sampler_config': {'_cls_name': 'EntropyBoundSamplerConfig',
  'entropy_bound': 0.1},
 'stability_threshold': 1,
 't_max': 0.8,
 't_min': 0.4,
 'transformers_version': '5.8.0.dev0'}

In [ ]:
gen_config # describes how to sample from the model

{'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 2,
 'dtype': 'bfloat16',
 'eos_token_id': 1,
 'final_logit_softcapping': 30.0,
 'global_head_dim': 512,
 'head_dim': 256,
 'hidden_activation': 'gelu_pytorch_tanh',
 'hidden_size': 2816,
 'initializer_range': 0.02,
 'intermediate_size': 2112,
 'layer_types': ['sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'full_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'full_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'full_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'full_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'sliding_attention',
  'full_attention

In model_config.json,  'eos_token_id': [1, 106]

In generation_config.json, eos_token_id': [1, 106, 50], token 50 is <|tool_response>. 1 and 106 are <eos> and <turn|> respectively.

Llama-3 instruct adds <|eot_id|> to its generation config the same way

In [ ]:
sd = {}
for safetensor_path in glob.glob(f"{checkpoint}/model-*.safetensors"):
    sd |= {k: v for k, v in load_file(safetensor_path).items() if "vision" not in k} # Diffusiongemma can also take images, but we are only concerned with text

['model.decoder.embed_tokens.weight', 'model.decoder.layers.0.experts.down_proj', 'model.decoder.layers.0.experts.gate_up_proj', 'model.decoder.layers.0.input_layernorm.weight', 'model.decoder.layers.0.layer_scalar', 'model.decoder.layers.0.mlp.down_proj.weight', 'model.decoder.layers.0.mlp.gate_proj.weight', 'model.decoder.layers.0.mlp.up_proj.weight', 'model.decoder.layers.0.post_attention_layernorm.weight', 'model.decoder.layers.0.post_feedforward_layernorm.weight', 'model.decoder.layers.0.post_feedforward_layernorm_1.weight', 'model.decoder.layers.0.post_feedforward_layernorm_2.weight', 'model.decoder.layers.0.pre_feedforward_layernorm.weight', 'model.decoder.layers.0.pre_feedforward_layernorm_2.weight', 'model.decoder.layers.0.router.per_expert_scale', 'model.decoder.layers.0.router.proj.weight', 'model.decoder.layers.0.router.scale', 'model.decoder.layers.0.self_attn.k_norm.weight', 'model.decoder.layers.0.self_attn.k_proj.weight', 'model.decoder.layers.0.self_attn.o_proj.weight'

In [ ]:
# sorted(list(sorted(sd.keys()))[:693:])
sorted(list(sorted(sd.keys()))) # all the weights we will use

['model.decoder.embed_tokens.weight',
 'model.decoder.layers.0.experts.down_proj',
 'model.decoder.layers.0.experts.gate_up_proj',
 'model.decoder.layers.0.input_layernorm.weight',
 'model.decoder.layers.0.layer_scalar',
 'model.decoder.layers.0.mlp.down_proj.weight',
 'model.decoder.layers.0.mlp.gate_proj.weight',
 'model.decoder.layers.0.mlp.up_proj.weight',
 'model.decoder.layers.0.post_attention_layernorm.weight',
 'model.decoder.layers.0.post_feedforward_layernorm.weight',
 'model.decoder.layers.0.post_feedforward_layernorm_1.weight',
 'model.decoder.layers.0.post_feedforward_layernorm_2.weight',
 'model.decoder.layers.0.pre_feedforward_layernorm.weight',
 'model.decoder.layers.0.pre_feedforward_layernorm_2.weight',
 'model.decoder.layers.0.router.per_expert_scale',
 'model.decoder.layers.0.router.proj.weight',
 'model.decoder.layers.0.router.scale',
 'model.decoder.layers.0.self_attn.k_norm.weight',
 'model.decoder.layers.0.self_attn.k_proj.weight',
 'model.decoder.layers.0.self_

In [ ]:
sd 

{'model.decoder.layers.17.experts.down_proj': tensor([[[-0.0486,  0.0132,  0.0199,  ..., -0.0220, -0.0576, -0.0266],
          [ 0.0080, -0.0981, -0.0079,  ...,  0.0510, -0.0017,  0.0120],
          [-0.0356,  0.0549,  0.0115,  ..., -0.0623,  0.0776,  0.0085],
          ...,
          [ 0.0057, -0.0003,  0.0182,  ...,  0.0032,  0.0474,  0.0167],
          [-0.0245, -0.0199, -0.0598,  ...,  0.0781,  0.0383,  0.0109],
          [ 0.0967, -0.0398,  0.0304,  ..., -0.0471, -0.0272, -0.0006]],
 
         [[ 0.0303, -0.0742,  0.0155,  ..., -0.0005,  0.0040,  0.0187],
          [ 0.0140, -0.1133, -0.0147,  ...,  0.0583,  0.0150, -0.0664],
          [-0.0092, -0.0518,  0.0254,  ..., -0.0742,  0.0491, -0.0732],
          ...,
          [-0.0280, -0.0449, -0.0215,  ..., -0.0055,  0.0515, -0.0160],
          [ 0.0138,  0.0427,  0.0126,  ..., -0.0020, -0.0405, -0.0540],
          [ 0.0074, -0.0408,  0.0481,  ..., -0.1016,  0.1089,  0.0361]],
 
         [[-0.0209, -0.0081, -0.0527,  ...,  0.0159,  0

Interestingly, all the Q/K norm weights are equal. This is likely by design. Note 

<q, k> = || q || * || k || * cos(theta),

If QK norm weights are all equal, then for all k1, k2 in R^n,

||k_norm_google(k1)|| = ||k_norm_google(k2)|| = k_norm_w

And likewise for q's. 

Then, note that since

attn_score_q (k_i) \proportional to e^(<q, k_i>)

\proportional to e^(cos(theta) / t)

where t = 1 / (q_norm_w * k_norm_w), a constant temperature on the attention scores.

With traditional QK-norm or non QK-norm models, ||q|| and ||k|| vary, thus we may have variable temperature per-query, and <q, k> won't only capture *relative importance* between q and k (cos theta), but also a measure of absolute importance of k (||k||) which is independent of the query. This may be undesirable for e.g., long-context prompts as an early key with large ||k|| that ceases to be important can still get an erroneously high score due to its magnitude.

In [11]:
W_vocab = sd['model.decoder.embed_tokens.weight']
print(W_vocab.shape, model_config['hidden_size'])
print(W_vocab.dtype)

torch.Size([262144, 2816]) 2816
torch.bfloat16


In [12]:
embed_scale = torch.tensor(model_config['hidden_size'] ** 0.5)

We do norm in fp32 as floating point errors accumulated through the sum is amplified as it is the denominator

In [13]:
def rms(x, w = 1):
    return (w * x / (torch.norm(x, dim=-1, keepdim=True, dtype=torch.float32) / (x.shape[-1] ** 0.5) + model_config['rms_norm_eps'])).to(x.dtype) 

# Precompute Rope

#### Partial rope

partial_factor = 0.25

In traditional rope, we linearly interpolate in log space with 

arange(0, head_dim, 2)

the ith wavelength is

theta ** (floor(i/2) * 2 / head_dim)

in proportional rope, we instead compress this into only partial_factor * head_dim many dimensions, i.e., we have

arange(0, head_dim, 2 / partial_factor)

the ith wavelength is

theta ** (floor(i/2) * 2 / (head_dim))

while leaving the remaining (1 - partial_factor) * head_dim nope

the important distinction is that we are doing

theta ** (floor(i/2) * 2 / (head_dim)), but only only partial_factor * head_dim many dimensions

In [14]:
print(model_config['head_dim'])
print(model_config['global_head_dim'])

256
512


In [ ]:
# precompute frequencies
freq = {}

# sliding window - regular rope
freq['sliding_attention'] = model_config['rope_parameters']['sliding_attention']['rope_theta'] ** -(torch.arange(0, 1, 2 / model_config['head_dim'], dtype=torch.float32))
# full attention - partial rope
freq['full_attention'] = model_config['rope_parameters']['full_attention']['rope_theta'] ** -(torch.arange(0, 1, 2 / model_config['global_head_dim'], dtype=torch.float32))
freq['full_attention'][int(model_config['rope_parameters']['full_attention']['partial_rotary_factor'] * len(freq['full_attention'])) : ] = 0

def rotate(x, layer_type, start_idx=0): # x [..., seq, head_dim]; rotate each (t, t + hd/2) pair ccw
    head_dim = x.shape[-1]
    rot_x = rearrange([-x[..., head_dim // 2 :], x[..., : head_dim // 2]], 'z ... d -> ... (z d)')
    pos = (torch.arange(x.shape[-2])[:, None] + start_idx) * torch.cat([freq[layer_type], freq[layer_type]]) # pos * (t, t + hd/2 pairs)
    return torch.cos(pos).to(x.dtype) * x + torch.sin(pos).to(x.dtype) * rot_x # apply rotation matrix; cast cos/sin so bf16 x isn't silently promoted to fp32

# Norms:

Per-layer, the we have the following norms:

┌──────────────────────────────┬──────────────────┬─────────────────────────────────────────────────────────┐
│            Weight            │    Component     │                   What it normalizes                    │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ input_layernorm              │ attention        │ block input → attention input (pre-attn)                │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ post_attention_layernorm     │ attention        │ attention output, before residual add (post-attn)       │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ pre_feedforward_layernorm    │ dense MLP branch │ block-post-attn residual → dense MLP input              │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ post_feedforward_layernorm_1 │ dense MLP branch │ dense MLP output                                        │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ pre_feedforward_layernorm_2  │ MoE branch       │ residual → MoE expert input │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ post_feedforward_layernorm_2 │ MoE branch       │ MoE output                                              │
├──────────────────────────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ post_feedforward_layernorm   │ merge            │ the sum h1 + h2, before residual add                    │
└──────────────────────────────┴──────────────────┴─────────────────────────────────────────────────────────┘ 

# Attention

In the "decode" stage, the attention for diffusionGemma is 

- similar to prefill in the sense that it writes kv's for several tokens at a time, decode in the sense that it reads a context history of past kv's

- similar to cross attention in that it attends to k,v's from the encoder, whisper-style, similar to self-attention in that it is non-casual, ViT-style

- simlar to the fused full-attention + cross-attention in MMDiT / Stable Diffusion 3, except the KV's for cross-attention is generated via casual instead of full attention.

In the "encode" stage, the attention is exactly the same as if the "verify" pass for speculative decoding with a 256 speculated tokens, with all tokens accepted.

This code can be easily modified to support Batch Size > 1, but not dynamic encode / decode within the same batch. Dynamic per-sequence attention style in supported in [VLLM](https://vllm-project.github.io/2026/06/10/diffusion-gemma.html):

![Vllm per seq masking](https://vllm-project.github.io/assets/figures/2026-06-10-diffusion-gemma/per_seq_causal_attention.svg)

("denoise" = decode, "prefill" = encode on prompt, "accept" = encode on denoised canvas). Our sliding window attention matches huggingface's implementation and is differernt from VLLM's, each canvas token attends to at most W tokens before the front of the canvas.

Note that we drop the / sqrt(N) term typically used in scaled dot product attention. Note this might have been aborbed into q_norm_w or k_norm_w.

Instead, Gemma uses an "embedding scale" that applies a scalar multiply by sqrt(D) at the start of the residual stream right before the first layer. I'm not sure why this is done.

Diffusiongemma unit-normalizes the value vector.

DiffusionGemma uses sliding window attention : global attention at a 5 : 1 ratio, with partial rope only on full attention. Both have the same number of q-heads, but full-attention has fewer gqa-groups / kv-heads.

The full attention layers also use KV sharing on global, so that before the rope rotation and k/v norms, they are equal. This is strange as k/v are typically believe to live in different subspaces (link), I wonder if the v_norm is what enables this. This let's us store only half the KV-cache for full-attention, which combined with hybrid full/sliding attention, makes diffusionGemma especially effective for long-context / KV-load memory bound regimes. This optimization is not currently implemented.

An optimization implemented is that on encode, we can skip all the computation after the k/v projection in the attention block in the final layer. This is because on encode passes, we don't care about the model's final output, we just need the minimum set of computation that will let us obtain the correct k/v's in all the layers, that we can commit to cache.

In the current implementation, full attention behaves like sliding window attention with max_tot_tokens sized window -- only the maximum kv cache size (kv_len) is relevant

In [ ]:
class AttentionBlock(torch.nn.Module):

    def __init__(self, layer_id):
        super().__init__()
        
        self.layer_id = layer_id
        self.layer_type = model_config['layer_types'][layer_id]
        self.W_q, self.W_k, self.W_o = [sd[f'model.decoder.layers.{layer_id}.self_attn.{item}_proj.weight'] for item in ['q', 'k', 'o']]

        self.q_norm, self.k_norm = [sd[f'model.decoder.layers.{layer_id}.self_attn.{item}_norm.weight'] for item in ['q', 'k']]
        self.pre_norm = sd[f'model.decoder.layers.{layer_id}.input_layernorm.weight']
        self.post_norm = sd[f'model.decoder.layers.{layer_id}.post_attention_layernorm.weight']

        self.q_heads = model_config['num_attention_heads']

        if self.layer_type == 'full_attention':
            self.W_v = self.W_k # global layers share K = V

            self.kv_heads = model_config['num_global_key_value_heads']
            self.head_dim = model_config['global_head_dim']

            self.kv_len = max_tot_tokens
        else:
            assert(self.layer_type == 'sliding_attention')
            self.W_v = sd[f'model.decoder.layers.{layer_id}.self_attn.v_proj.weight']

            self.kv_heads = model_config['num_key_value_heads']
            self.head_dim = model_config['head_dim']

            self.kv_len = model_config['sliding_window']

        self.k_cache = torch.empty(self.kv_heads, self.kv_len, self.head_dim) # use statically shaped KV buffer
        self.v_cache = torch.empty(self.kv_heads, self.kv_len, self.head_dim) # KV's flow into here from left to right, FIFO, latest element is rightmost
            

    def forward(self, x, pos_idx, mode):
        assert mode in ['encode', 'decode']

        L, D = x.shape
        print(L, D)

        resid_x = x.clone()
        
        x = rms(x, w=self.pre_norm)
        
        q, k, v = x @ self.W_q.T, x @ self.W_k.T, x @ self.W_v.T
        q, k, v = [rearrange(z, 'l (n h) -> n l h', h = self.head_dim) for z in [q, k, v]]
        q, k = rms(q, self.q_norm), rms(k, self.k_norm) # QK-norm per head: weight is [head_dim], normalize over each head's dims
        v = rms(v, 1) # v_norm: weightless, no rope

        q, k = rotate(q, self.layer_type, pos_idx), rotate(k, self.layer_type, pos_idx) # absolute positions pos_idx .. pos_idx + L
        
        k = torch.concat([self.k_cache[:, : pos_idx, :], k], axis=1) # attend to [committed history | current block]
        v = torch.concat([self.v_cache[:, : pos_idx, :], v], axis=1) # Note python automatically clips on the left to 0, on the right to shape[1] = kv_len
        
        if mode == "encode": # difference #1: writes/updates the kv cache
            # Actually, we don't have to put this in a branch, can also just do this on decode too, ok since we'll override with an encode at the end anyways
            self.k_cache[:, : pos_idx + L, :] = k[:, -self.kv_len :, :] # automatically clips
            self.v_cache[:, : pos_idx + L, :] = v[:, -self.kv_len :, :]

            if self.layer_id == model_config['num_hidden_layers'] - 1:
                return # encode optimization: notice we don't need to do the remaining computation after this

        q = rearrange(q, '(n gqa) qt h -> n gqa qt h', gqa = self.q_heads // self.kv_heads) # Fold GQA into an outer dim

        scores = einsum(q, k, 'n gqa qt h, n kt h -> n gqa qt kt').float() # no divide by sqrt(head dim); softmax in fp32
        
        if mode == "encode":
            scores += torch.triu(torch.full(scores.shape, -torch.inf), diagonal = scores.shape[-1] - scores.shape[-2] + 1) # this applies a mask that looks like R2 in the vllm figure
        
        scores = torch.exp(scores - torch.amax(scores, axis=-1, keepdims=True))
        scores = (scores / torch.sum(scores, axis=-1, keepdims=True)).to(x.dtype)

        x = einsum(scores, v, 'n gqa qt kt, n kt h -> n gqa qt h')
        x = rearrange(x, 'n gqa qt h -> qt (n gqa h)')

        res = x @ self.W_o.T
        res = rms(res, self.post_norm)

        return res + resid_x

In [17]:
sd['model.decoder.layers.0.experts.gate_up_proj'].shape


torch.Size([128, 1408, 2816])

In [18]:
model_config['moe_intermediate_size']

# I assumed running through Slurm (no ssh, no kill) meant I couldn't disrupt

704

There is an implementation trick we can do here to save a layernorm, by observing self.preNorm in weight

do a get_weight thingy

index_add_ is "scatter-add": it takes a batch of rows, and adds each one into a destination tensor at a position you specify — accumulating when two rows target the same position. Here's it from the ground up.

The signature

dest.index_add_(dim, index, source)

- The trailing underscore means in-place: it modifies dest directly (and returns it).
- dim — which axis of dest you're indexing into (almost always 0 in MoE code: the token axis).
- index — a 1-D integer tensor saying where each row of source should go.
- source — the values to add. Must have the same shape as dest except along dim, where its length must equal len(index).

It is exactly equivalent to this loop (for dim=0):

for i in range(len(index)):
    dest[index[i]] += source[i]     # note: +=, not =

A tiny example

import torch

dest   = torch.zeros(4, 3)                  # 4 slots, each a 3-dim vector
index  = torch.tensor([2, 0, 2])            # where each source row goes
source = torch.tensor([[1., 1., 1.],
                       [5., 5., 5.],
                       

In [19]:
class MLP(torch.nn.Module):
    def __init__(self, layer_id, expert_num, conditioning_mlp : bool = False):
        super().__init__()

        if(conditioning_mlp):
            self.pre_norm = sd['model.decoder.self_conditioning.pre_norm.weight']

            self.W_up = sd['model.decoder.self_conditioning.up_proj.weight']
            self.W_gate = sd['model.decoder.self_conditioning.gate_proj.weight']
            self.W_down = sd['model.decoder.self_conditioning.down_proj.weight']
            return

        if expert_num is None:
            self.pre_norm = sd[f'model.decoder.layers.{layer_id}.pre_feedforward_layernorm.weight']
            self.W_up, self.W_gate, self.W_down = [sd[f'model.decoder.layers.{layer_id}.mlp.{item}_proj.weight'] for item in ['up', 'gate', 'down']]
        else:
            self.pre_norm = sd[f'model.decoder.layers.{layer_id}.pre_feedforward_layernorm_2.weight']
            
            self.W_gate, self.W_up = rearrange(sd[f'model.decoder.layers.{layer_id}.experts.gate_up_proj'][expert_num], '(z intermed) D -> z intermed D', z=2)
            self.W_down = sd[f'model.decoder.layers.{layer_id}.experts.down_proj'][expert_num]
    
    def forward(self, x): # pre-norm + gated MLP only; the post-norms live at the call sites (MOEBlock / self-conditioning add)
        L, D = x.shape

        x = rms(x, self.pre_norm)
        
        a = x @ self.W_up.T
        b = F.gelu(x @ self.W_gate.T)
        res = (a * b) @ self.W_down.T

        assert(res.shape == (L, D)) # both shared and expert maps (_, D) -> (_, D)

        return res

In [20]:
a = MLP(layer_id=None, expert_num=None, conditioning_mlp=True)

For batch > 1, we would need to do an AlltoAll (https://jax-ml.github.io/scaling-book/sharding/) to get, for each expert, all the 

In [21]:
class MOEBlock(torch.nn.Module):
    def __init__(self, layer_id):
        super().__init__()
        self.k_experts = model_config['top_k_experts']
        self.num_experts = model_config['num_experts']

        self.W_router = sd[f'model.decoder.layers.{layer_id}.router.proj.weight']
        self.expert_scale = sd[f'model.decoder.layers.{layer_id}.router.per_expert_scale']
        self.scale = sd[f'model.decoder.layers.{layer_id}.router.scale']

        self.experts = [MLP(layer_id, e) for e in range(self.num_experts)]
        self.shared_expert = MLP(layer_id, None)
        
        self.post_norm_1 = sd[f'model.decoder.layers.{layer_id}.post_feedforward_layernorm_1.weight'] # dense MLP output
        self.post_norm_2 = sd[f'model.decoder.layers.{layer_id}.post_feedforward_layernorm_2.weight'] # aggregated MoE output, applied ONCE (not per expert)
        self.post_norm = sd[f'model.decoder.layers.{layer_id}.post_feedforward_layernorm.weight'] # the sum h1 + h2, before residual add

    def forward(self, x):
        L, D = x.shape

        resid_x = x.clone()

        route_x = rms(x) * self.scale / (D ** 0.5)

        expert_scores = F.softmax(route_x @ self.W_router.T, dim=-1) # (L, num_experts)

        top_k_scores, top_k_idx = torch.topk(expert_scores, self.k_experts, dim=-1) # (L, k_experts), (L, k_experts)
        top_k_scores = top_k_scores / torch.sum(top_k_scores, dim=-1, keepdim=True) * self.expert_scale[top_k_idx] # normalize to sum to 1, THEN scale per expert (multiply, not divide)

        res = rms(self.shared_expert(x), self.post_norm_1) # h1: dense branch, shape (L, D)

        print(top_k_idx.shape, top_k_scores.shape)

        moe_out = torch.zeros_like(x)
        for id, expert in zip(range(self.num_experts), self.experts):
            mask = torch.any(top_k_idx == id, dim = -1) # boolean mask (L, ) which tokens routed to expert_id
            mult = top_k_scores[mask][top_k_idx[mask] == id] # shape (L', ) where L' <= L is the number of tokens expert_id routed to
            moe_out[mask] += mult[:, None] * expert(x[mask]) # (L', D) += (L', 1) * (L', D)

        res = res + rms(moe_out, self.post_norm_2) # h2 normed once, then h1 + h2

        return rms(res, self.post_norm) + resid_x


In [22]:
# temp = MOEBlock(0)

# x = torch.ones([5, model_config['hidden_size']], dtype=torch.bfloat16)

# temp(x)

Explain logit softcapping

In [ ]:
class DiffusionGemma(torch.nn.Module): # Any computation that utilizes parameters passes through here

    def __init__(self):
        super().__init__()

        self.W_embed = sd['model.decoder.embed_tokens.weight'] # also used as the unembedding matrix (tie_word_embeddings = True)

        self.attn_blocks = [AttentionBlock(i) for i in range(model_config['num_hidden_layers'])]
        self.moe_blocks = [MOEBlock(i) for i in range(model_config['num_hidden_layers'])]

        self.multiplier_enc = [sd[f'model.encoder.language_model.layers.{i}.layer_scalar'] for i in range(model_config['num_hidden_layers'])]
        self.multiplier_dec = [sd[f'model.decoder.layers.{i}.layer_scalar'] for i in range(model_config['num_hidden_layers'])]

        self.model_norm = sd['model.decoder.norm.weight'] # final / model norm

        self.conditioning_MLP = MLP(layer_id=None, expert_num=None, conditioning_mlp=True)

    def _forward(self, pos_idx, logits, logit_probs, mode): # logit_probs = 0 <=> skip this path, since a composition of functions that map 0 to 0 still maps 0 to 0 (no bias term anywhere)
        x = self.W_embed[logits] * embed_scale # (L, ) -> (L, D)

        # do self conditioning if decode
        if mode == "decode":
            condition_x = (logit_probs.to(x.dtype) @ self.W_embed) * embed_scale # (L, V) x (V, D) --> convex combination of vocab embeddings
            condition_x = self.conditioning_MLP(condition_x)
            x = rms(x + condition_x)

        # pass through all layers
        for i, (attn, moe) in enumerate(zip(self.attn_blocks, self.moe_blocks)):
            x = attn(x, pos_idx, mode)
            if x is None: # last encode layer wrote its KV cache and returned early; nothing else is needed
                return None
            x = moe(x)
            x = x * (self.multiplier_enc if mode == "encode" else self.multiplier_dec)[i] # per-layer encoder/decoder scalar

        x = rms(x, self.model_norm)

        final_logits = (x @ self.W_embed.T).float() # (L, D) x (D, V) -> (L, V)

        return torch.tanh(final_logits / model_config['final_logit_softcapping']) * model_config['final_logit_softcapping']
        
    # this commits canvas, (last time we) write KV cache
    def enc(self, pos_idx, logits) -> None: 
        self._forward(pos_idx, logits, 0, mode="encode")

    # this maps (canvas_i, canvas_prob_i) -> (canvas_prob_i+1)
    def dec(self, pos_idx, logits, logit_probs) -> torch.tensor:
        return self._forward(pos_idx, logits, logit_probs, mode="decode")

https://ai.google.dev/gemma/docs/diffusiongemma -- sampling parametrrs

In [24]:
prompt = "What does 67 mean?"

tok = Tokenizer.from_file(f"{checkpoint}/tokenizer.json")
chat = f"<bos><|turn>user\n{prompt}<turn|>\n<|turn>model\n"
ids = tok.encode(chat, add_special_tokens=False).ids


In [25]:
max_denoising_steps = gen_config['max_denoising_steps']      # decoder forward passes per canvas (generation_config.json default)
entropy_bound = gen_config['sampler_config']['entropy_bound']           # accept tokens with entropy <= this; larger accepts more tokens per step
t_max, t_min = gen_config['t_max'], gen_config['t_min']       # sampling temperature anneals t_max -> t_min across the steps
confidence_threshold = gen_config['confidence_threshold']  # early-stop a canvas when argmax is stable and mean entropy < this

print_freq = 4                # print the current draft every print_freq steps

In [26]:
gen_config

# what is stability threshold?

{'confidence_threshold': 0.005,
 'eos_token_id': [1, 106, 50],
 'max_denoising_steps': 48,
 'max_new_tokens': 256,
 'pad_token_id': 0,
 'sampler_config': {'_cls_name': 'EntropyBoundSamplerConfig',
  'entropy_bound': 0.1},
 'stability_threshold': 1,
 't_max': 0.8,
 't_min': 0.4,
 'transformers_version': '5.8.0.dev0'}

![Image of Yaktocat](https://vllm-project.github.io/assets/figures/2026-06-10-diffusion-gemma/sampling-loop-horizontal.svg)

Google should make the first step condition on a random nroamlized uniform distribution, then denoising / self-conditioning has a cleaner interpretation as iterative distribution shaping

Technically, the only information that needs to be passed between successive denoising passes is the vocab logprobs. 

We can do the sample only when we need to, instead of right away

bag of tricks for speeding up diffusiongemma inference

small improvement: don't pass full logprobs for conditioning

what if we don't pass logits at all? Only pass logporbs, nothign is committed until it is committed

overlapping block diffusion? note the conditioning mlp is on D only. 

Also, due to extreme sparseness, waayyy compress the thigns we pass between two iterations lmao



In [27]:
tokens = torch.tensor(tok.encode(chat, add_special_tokens=False).ids) # TODO: change to compressed logits (since they're sparse)

print(tokens)

tensor([     2,    105,   2364,    107,   3689,   1677, 236743, 236825, 236832,
          2689, 236881,    106,    107,    105,   4368,    107])


In [28]:
V = model_config['vocab_size']

In [29]:
model = DiffusionGemma()


# Sampling

In the docs, each denoising step is described as:

- model.decode forward
- retain logits for self-conditioning
- sample with renoising

And the inital canvas is random logits with all logits = 0, so the self-conditioning MLP maps all 0 to all 0s (zero contribution from self-conditioning)

Here, we restructure slightly so each denoising step first samples with renoising from the previous canvas, then computes the logits for the current iteration. This is nice since:
- The only information we need to pass between steps is the logprobs. We do the sampling when we need it, lazily.
- We can fold the initial canvas initization into the first denoising step, by just passing in all equal logits
- More naturally yields to the interpretation of self-conditioning not as an additional input to the decoder, but as a first-class primative -- the decoder steadily shapes the logprobs guided by sampling. To this end, another design for diffusiongemma could feed in a torch.rand()/sum(torch.rand()) as the pure noise logits input to be shaped from scratch, instead of 0 (meaning no self-conditioning)

We also intentially give general functions for denoise and commit to make it easier to explore alternatives to standard block diffusion

denoise is a pure function that has no side effects, so we can denoise multiple times and choose the best one, we can also denoise anywhere, as long as pos_idx <= len(tokens) - canvas_len. The correct slice of the KV cache will automatically be read, which is the only information the model gets about the history before pos_idx.

commit applies encode and writes (overwrites) the slide of KV cache from l to r with the corresponding KV's for staged tokens. This is exactly the same computation we do in the "verify" step in speculative decoding for regular LLMs. So if we're overwriting and staged tokens != prev_canvas[l:r+1], all of the KV_cache[r+1:] is invalidated. However, the ability to commit and denoise flexibally lets us explore different research ideas. Due to decode requiring self-attention with exactly canvas_len number of tokens, we can denoise arbitrarily-lengthed blocks, but the blocks we denoise can, for example, overlap!

The renoising and stopping condition are exactly the same as the huggingface implementation:

- After the current denoising step, we look at the canvas of logits. Each position in the canvas has some associated entropy (of the post softmax distribution). We select the maximum number of positions such that the sum of their entropies is <= some threshold, and renoise (uniform sampling from vocab) the rest, breaking ties by choosing the set of positions that has smaller cumulative entropy. 

- We use a linear schedule of decreasing tempeatures. As the docs describes "..."

- We stop when the mean entropy in the current canvas is less than some threshold, and the argmax of current canvas = argmax of previous canvas

In [30]:
def denoise(pos_idx): # returns staged_tokens
    assert pos_idx + canvas_len <= len(tokens) # must be length canvas_len (what if it wasn't fixed? Analyze the self attention

    t = t_max
    t_step = (t_min - t_max) / max_denoising_steps

    last_canvas = Categorical(logits = torch.ones((canvas_len, V)))

    for step in trange(max_denoising_steps):

        # renoise last_canvas
        sH, sidx = last_canvas.entropy().sort(-1)
        accepted = torch.zeros_like(sH, dtype=torch.bool).scatter(-1, sidx, sH.cumsum(-1) - sH <= entropy_bound)
        last_canvas_noised = torch.where(accepted, last_canvas.sample(), torch.randint(0, V, (canvas_len,)))

        # pass in the noised tokens, but un-noised normalized probs (all-zero probs on step 0: the conditioning path maps 0 to 0)
        canvas = model.dec(pos_idx, last_canvas_noised, last_canvas.probs if step != 0 else torch.zeros(canvas_len, V)) / t
        canvas = Categorical(logits = canvas)

        if torch.mean(canvas.entropy()) < confidence_threshold and (canvas.logits.argmax(dim=-1) == last_canvas.logits.argmax(dim=-1)).all():
            return canvas.sample()
        
        t += t_step
        last_canvas = canvas

    assert False, f"Denoising not finished after {max_denoising_steps} steps"

def commit(l, r, staged_tokens):
    global tokens
    assert len(staged_tokens) == r - l + 1

    if(r + 1 < len(tokens)): # commits staged_tokens
        print(f"Invalidating {len(tokens) - (r+1)} tokens")
        tokens = tokens[:r+1]
    
    tokens[l:] = staged_tokens
    model.enc(l, staged_tokens)

def new_canvas():
    global tokens
    if len(tokens) + canvas_len > max_tot_tokens:
        return False
    
    nxt = torch.randint(V, (canvas_len,))
    tokens = torch.concat([tokens, nxt])
    return True

In [ ]:
commit(0, len(tokens) - 1, tokens) # prefill: encode the chat prompt into the KV cache
pos_idx = len(tokens)

while new_canvas():
    staged_tokens = denoise(pos_idx)
    commit(pos_idx, pos_idx + canvas_len - 1, staged_tokens)
    pos_idx = len(tokens)

    print("\nMODEL OUTPUT:\n" + tok.decode(tokens.tolist()))

    if torch.isin(staged_tokens, torch.tensor(gen_config['eos_token_id'])).any():
        break

16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([16, 8])
16 2816
torch.Size([16, 8]) torch.Size([

  0%|          | 0/48 [00:00<?, ?it/s]

256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) t

  2%|▏         | 1/48 [39:10<30:41:29, 2350.84s/it]

256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) t

  4%|▍         | 2/48 [1:21:35<31:29:33, 2464.65s/it]

256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) torch.Size([256, 8])
256 2816
torch.Size([256, 8]) t

  6%|▋         | 3/48 [2:07:38<32:30:51, 2601.15s/it]

256 2816
torch.Size([256, 8]) torch.Size([256, 8])
